# 🗄️ Week 6 — SQL & Databases Practice

## Working with SQLite and Pandas

**Welcome to the SQL portion of Week 6!** In this notebook, you'll learn to:

---

### 🎯 Learning Objectives

By the end of this session, you will be able to:

- ✅ Connect to SQLite databases using Python
- ✅ Execute SQL queries (SELECT, WHERE, ORDER BY)
- ✅ Perform JOIN operations to combine tables
- ✅ Use GROUP BY for aggregation
- ✅ Load SQL results into Pandas DataFrames
- ✅ Combine SQL and Pandas for data analysis

### 📋 Database Schema Overview

We'll work with a **Music Database** containing:

| Table | Description | Key Columns |
|-------|-------------|-------------|
| `artists` | Music artists | artist_id, name, country, genre |
| `albums` | Albums by artists | album_id, title, artist_id, release_year |
| `tracks` | Individual songs | track_id, title, album_id, duration_seconds |
| `playlists` | User playlists | playlist_id, name, description |
| `playlist_tracks` | Many-to-many junction | playlist_id, track_id, position |
| `album_sales` | Sales records | sale_id, album_id, quantity, unit_price, region |

**💡 Real-world parallel:** This is similar to how Spotify or Apple Music stores their data!

---

## 🚀 Setup: Initialize the Test Database

First, let's run the setup script to create our test database from the SQL file.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🚀 SETUP: Create the test database (works in Colab & local Jupyter)
# ═══════════════════════════════════════════════════════════════════════════════

import sqlite3
import os
import subprocess
from pathlib import Path

# Check if we're in Google Colab
IN_COLAB = False
try:
    import google.colab  # type: ignore  # Only available in Colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("☁️  Running in Google Colab - setting up environment...")
    
    # Clone the repo if not already present
    if not os.path.exists('Python-Essentials-Code-The-Dream'):
        subprocess.run(['git', 'clone', 'https://github.com/StrayDogSyn/Python-Essentials-Code-The-Dream.git'], check=True)
        print("✅ Repository cloned!")
    
    # Change to week8 directory
    os.chdir('Python-Essentials-Code-The-Dream/notebook-sessions/week8')
    print(f"📁 Working directory: {os.getcwd()}")
else:
    print("💻 Running locally")

# Now set up the database
DB_PATH = "test.db"
SQL_PATH = "test.sql"

# Check if SQL file exists
if os.path.exists(SQL_PATH):
    # Remove existing database
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print("🗑️  Removed existing database")
    
    # Read and execute SQL
    with open(SQL_PATH, 'r', encoding='utf-8') as f:
        sql_script = f.read()
    
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript(sql_script)
        
        # Verify tables
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
        tables = cursor.fetchall()
        
    print("═" * 60)
    print("✅ Database created successfully!")
    print(f"📊 Tables: {', '.join([t[0] for t in tables])}")
    print("═" * 60)
else:
    print("⚠️  SQL file not found. Make sure you're in the week8 directory.")
    print(f"   Looking for: {os.path.abspath(SQL_PATH)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📦 IMPORTS: Required libraries
# ═══════════════════════════════════════════════════════════════════════════════

import sqlite3
import pandas as pd
from pathlib import Path

# Path to our database
DB_PATH = "test.db"

print("✅ Libraries imported!")
print(f"📦 Pandas version: {pd.__version__}")
print(f"🗄️  Database: {DB_PATH}")

---

## 📊 Section 1: Connecting to SQLite

SQLite is a lightweight, file-based database that's built into Python. No server setup required!

### Connection Methods

| Method | Pros | Cons |
|--------|------|------|
| `sqlite3.connect()` | Direct control | Manual cleanup |
| `with sqlite3.connect()` | Auto-closes | Best practice! |
| `pd.read_sql()` | Returns DataFrame | Needs connection |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ⭐ BEST PRACTICE: Using context manager for connections
# ═══════════════════════════════════════════════════════════════════════════════

# The 'with' statement automatically closes the connection when done
with sqlite3.connect(DB_PATH) as conn:
    # Enable foreign key support
    conn.execute("PRAGMA foreign_keys = ON")
    
    # Create a cursor to execute queries
    cursor = conn.cursor()
    
    # List all tables in the database
    cursor.execute("""
        SELECT name FROM sqlite_master 
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """)
    
    tables = cursor.fetchall()
    
print("📁 Tables in our database:")
for (table,) in tables:
    print(f"   • {table}")

---

## 📊 Section 2: Basic SELECT Queries

The `SELECT` statement is the foundation of SQL. It retrieves data from tables.

### SELECT Syntax

```sql
SELECT column1, column2, ...  -- What columns to retrieve
FROM table_name               -- Which table to query
WHERE condition               -- Filter rows (optional)
ORDER BY column               -- Sort results (optional)
LIMIT n;                      -- Limit rows returned (optional)
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🌱 NOVICE: Basic SELECT - Get all artists
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    
    # SELECT * means "all columns"
    cursor.execute("SELECT * FROM artists")
    
    # Fetch all results
    results = cursor.fetchall()
    
    # Get column names from cursor description
    columns = [description[0] for description in cursor.description]

print(f"📊 Artists Table ({len(results)} rows)")
print(f"Columns: {columns}")
print("-" * 70)
for row in results:
    print(row)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ✨ PYTHONIC: Load directly into Pandas DataFrame
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    # pd.read_sql() is the magic bridge between SQL and Pandas!
    df_artists = pd.read_sql("SELECT * FROM artists", conn)

print("📊 Artists as DataFrame:")
df_artists

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔍 SELECT with WHERE clause - Filter results
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    # Get albums released after 2000
    query = """
        SELECT title, release_year, genre
        FROM albums
        WHERE release_year > 2000
        ORDER BY release_year DESC
    """
    df_modern = pd.read_sql(query, conn)

print("📀 Albums released after 2000:")
df_modern

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 YOUR TURN: Write a SELECT query
# ═══════════════════════════════════════════════════════════════════════════════
# Challenge: Get all albums from the 'Rock' genre, sorted by release_year

with sqlite3.connect(DB_PATH) as conn:
    # TODO: Write your query here
    query = """
        SELECT title, release_year, genre
        FROM albums
        WHERE genre = 'Rock'
        ORDER BY release_year
    """
    df_rock = pd.read_sql(query, conn)

print("🎸 Rock Albums:")
df_rock

---

## 🔗 Section 3: JOIN Operations

JOINs combine data from multiple tables. This is where SQL really shines!

### Types of JOINs

| JOIN Type | Returns |
|-----------|--------|
| `INNER JOIN` | Only matching rows from both tables |
| `LEFT JOIN` | All rows from left table + matches from right |
| `RIGHT JOIN` | All rows from right table + matches from left |
| `FULL OUTER JOIN` | All rows from both tables |

**💡 Most common:** `INNER JOIN` for finding related records

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔗 INNER JOIN: Combine albums with artist names
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT 
            albums.title AS album_title,
            artists.name AS artist_name,
            albums.release_year,
            albums.genre
        FROM albums
        INNER JOIN artists ON albums.artist_id = artists.artist_id
        ORDER BY albums.release_year
    """
    df_albums_artists = pd.read_sql(query, conn)

print("📀 Albums with Artist Names:")
df_albums_artists

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔗 Multiple JOINs: Tracks → Albums → Artists
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT 
            tracks.title AS track,
            albums.title AS album,
            artists.name AS artist,
            ROUND(tracks.duration_seconds / 60.0, 2) AS duration_min
        FROM tracks
        INNER JOIN albums ON tracks.album_id = albums.album_id
        INNER JOIN artists ON albums.artist_id = artists.artist_id
        ORDER BY tracks.duration_seconds DESC
    """
    df_tracks_full = pd.read_sql(query, conn)

print("🎵 Tracks with Album and Artist Info (sorted by duration):")
df_tracks_full

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🔗 LEFT JOIN: All artists, even those without albums
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT 
            artists.name AS artist,
            COUNT(albums.album_id) AS album_count
        FROM artists
        LEFT JOIN albums ON artists.artist_id = albums.artist_id
        GROUP BY artists.artist_id
        ORDER BY album_count DESC
    """
    df_artist_counts = pd.read_sql(query, conn)

print("📊 Albums per Artist:")
df_artist_counts

---

## 📈 Section 4: Aggregation with GROUP BY

Aggregation functions summarize data. Combined with `GROUP BY`, you can create powerful reports!

### Common Aggregation Functions

| Function | Purpose | Example |
|----------|---------|--------|
| `COUNT()` | Count rows | `COUNT(*)` |
| `SUM()` | Add values | `SUM(quantity)` |
| `AVG()` | Average | `AVG(price)` |
| `MIN()` | Minimum | `MIN(release_year)` |
| `MAX()` | Maximum | `MAX(duration)` |
| `ROUND()` | Round numbers | `ROUND(AVG(price), 2)` |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 GROUP BY: Aggregate album stats by genre
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT 
            genre,
            COUNT(*) AS num_albums,
            ROUND(AVG(duration_minutes), 1) AS avg_duration,
            MIN(release_year) AS earliest,
            MAX(release_year) AS latest
        FROM albums
        GROUP BY genre
        ORDER BY num_albums DESC
    """
    df_genre_stats = pd.read_sql(query, conn)

print("📊 Album Statistics by Genre:")
df_genre_stats

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 💰 Sales Analysis: Total revenue by region
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT 
            region,
            COUNT(*) AS num_transactions,
            SUM(quantity) AS total_units,
            ROUND(SUM(quantity * unit_price), 2) AS total_revenue
        FROM album_sales
        GROUP BY region
        ORDER BY total_revenue DESC
    """
    df_sales_region = pd.read_sql(query, conn)

print("💰 Sales by Region:")
df_sales_region

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🏆 Best-Selling Albums (JOIN + GROUP BY)
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT 
            albums.title AS album,
            artists.name AS artist,
            SUM(album_sales.quantity) AS units_sold,
            ROUND(SUM(album_sales.quantity * album_sales.unit_price), 2) AS revenue
        FROM album_sales
        INNER JOIN albums ON album_sales.album_id = albums.album_id
        INNER JOIN artists ON albums.artist_id = artists.artist_id
        GROUP BY albums.album_id
        ORDER BY revenue DESC
    """
    df_best_sellers = pd.read_sql(query, conn)

print("🏆 Best-Selling Albums:")
df_best_sellers

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 HAVING: Filter groups (like WHERE, but for aggregates)
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    # Find formats with more than $20 in total revenue
    query = """
        SELECT 
            format,
            COUNT(*) AS num_sales,
            ROUND(SUM(quantity * unit_price), 2) AS total_revenue
        FROM album_sales
        GROUP BY format
        HAVING total_revenue > 20
        ORDER BY total_revenue DESC
    """
    df_format_sales = pd.read_sql(query, conn)

print("📀 Sales by Format (>$20 revenue):")
df_format_sales

---

## 🔒 Section 5: Parameterized Queries (Security!)

**⚠️ CRITICAL:** Never use f-strings or string concatenation with user input in SQL queries!

This prevents **SQL Injection attacks** - one of the most common security vulnerabilities.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🚫 BAD PRACTICE: String formatting (NEVER do this with user input!)
# ═══════════════════════════════════════════════════════════════════════════════

print("🚫 BAD: String formatting in SQL")
print("-" * 50)

# Example of what NOT to do:
user_input = "Rock"
bad_query = f"SELECT * FROM albums WHERE genre = '{user_input}'"
print(f"Query: {bad_query}")
print()
print("❌ If user_input was: \"Rock'; DROP TABLE albums; --\"")
print("   The query would become: SELECT * FROM albums WHERE genre = 'Rock'; DROP TABLE albums; --'")
print("   This would DELETE your entire table!")
print("=" * 50)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ✅ BEST PRACTICE: Parameterized queries
# ═══════════════════════════════════════════════════════════════════════════════

print("✅ SAFE: Parameterized queries")
print("-" * 50)

# Safe way - use ? placeholders
user_input = "Rock"

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    
    # The ? is a placeholder that gets safely substituted
    safe_query = "SELECT title, release_year FROM albums WHERE genre = ?"
    cursor.execute(safe_query, (user_input,))  # Pass parameters as tuple
    
    results = cursor.fetchall()

print(f"Query: SELECT title, release_year FROM albums WHERE genre = ?")
print(f"Parameter: {user_input}")
print()
print("Results:")
for title, year in results:
    print(f"  • {title} ({year})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ✅ Parameterized queries with Pandas
# ═══════════════════════════════════════════════════════════════════════════════

# You can also use parameters with pd.read_sql()
genre = "Pop"
min_year = 2000

with sqlite3.connect(DB_PATH) as conn:
    query = """
        SELECT title, release_year, genre
        FROM albums
        WHERE genre = ? AND release_year >= ?
        ORDER BY release_year
    """
    df_filtered = pd.read_sql(query, conn, params=(genre, min_year))

print(f"📀 {genre} albums from {min_year} onwards:")
df_filtered

---

## 🐼 Section 6: SQL + Pandas Integration

The power combo! Use SQL to query, Pandas to analyze and visualize.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🐼 Load data with SQL, analyze with Pandas
# ═══════════════════════════════════════════════════════════════════════════════

with sqlite3.connect(DB_PATH) as conn:
    # Get all sales data with album info
    query = """
        SELECT 
            album_sales.*,
            albums.title AS album_title,
            artists.name AS artist_name,
            albums.genre
        FROM album_sales
        INNER JOIN albums ON album_sales.album_id = albums.album_id
        INNER JOIN artists ON albums.artist_id = artists.artist_id
    """
    df_sales = pd.read_sql(query, conn)

print(f"📊 Loaded {len(df_sales)} sales records")
print()
print("Columns:", list(df_sales.columns))
df_sales.head()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📊 Pandas aggregation on SQL data
# ═══════════════════════════════════════════════════════════════════════════════

# Add a calculated column
df_sales['revenue'] = df_sales['quantity'] * df_sales['unit_price']

# Group by genre using Pandas
genre_summary = df_sales.groupby('genre').agg({
    'quantity': 'sum',
    'revenue': ['sum', 'mean'],
    'sale_id': 'count'
}).round(2)

# Flatten column names
genre_summary.columns = ['total_units', 'total_revenue', 'avg_revenue', 'num_transactions']
genre_summary = genre_summary.sort_values('total_revenue', ascending=False)

print("📊 Sales Summary by Genre (Pandas aggregation):")
genre_summary

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📈 Quick visualization
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

# Revenue by region
region_sales = df_sales.groupby('region')['revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
region_sales.plot(kind='barh', color=['#00f0ff', '#ff00ff', '#00ff41'], ax=ax)
ax.set_xlabel('Revenue ($)')
ax.set_ylabel('Region')
ax.set_title('💰 Album Sales Revenue by Region')
ax.set_facecolor('#1a1f3a')
fig.patch.set_facecolor('#0a0e27')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('#00f0ff')
for spine in ax.spines.values():
    spine.set_color('#00f0ff')
plt.tight_layout()
plt.show()

print(f"📊 Total Revenue: ${df_sales['revenue'].sum():.2f}")

---

## 🎯 Practice Challenges

Test your SQL skills with these exercises!

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 CHALLENGE 1: Find all UK artists and their albums
# ═══════════════════════════════════════════════════════════════════════════════
# Hint: Use JOIN to combine artists and albums, filter WHERE country = 'UK'

with sqlite3.connect(DB_PATH) as conn:
    # TODO: Write your query
    query = """
        -- Your SQL here
        SELECT 
            artists.name AS artist,
            albums.title AS album,
            albums.release_year
        FROM artists
        INNER JOIN albums ON artists.artist_id = albums.artist_id
        WHERE artists.country = 'UK'
        ORDER BY artists.name, albums.release_year
    """
    df_uk = pd.read_sql(query, conn)

print("🇬🇧 UK Artists and Their Albums:")
df_uk

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 CHALLENGE 2: Calculate average album duration by decade
# ═══════════════════════════════════════════════════════════════════════════════
# Hint: Use (release_year / 10) * 10 to get decade, then GROUP BY

with sqlite3.connect(DB_PATH) as conn:
    # TODO: Write your query
    query = """
        -- Your SQL here
        SELECT 
            (release_year / 10) * 10 AS decade,
            COUNT(*) AS num_albums,
            ROUND(AVG(duration_minutes), 1) AS avg_duration
        FROM albums
        GROUP BY decade
        ORDER BY decade
    """
    df_decades = pd.read_sql(query, conn)

print("📅 Album Duration by Decade:")
df_decades

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🎯 CHALLENGE 3: Find the most popular playlist tracks
# ═══════════════════════════════════════════════════════════════════════════════
# Hint: Join playlist_tracks → tracks → albums → artists, count appearances

with sqlite3.connect(DB_PATH) as conn:
    # TODO: Write your query
    query = """
        -- Your SQL here
        SELECT 
            tracks.title AS track,
            artists.name AS artist,
            COUNT(playlist_tracks.playlist_id) AS playlist_count
        FROM playlist_tracks
        INNER JOIN tracks ON playlist_tracks.track_id = tracks.track_id
        INNER JOIN albums ON tracks.album_id = albums.album_id
        INNER JOIN artists ON albums.artist_id = artists.artist_id
        GROUP BY tracks.track_id
        ORDER BY playlist_count DESC
    """
    df_popular_tracks = pd.read_sql(query, conn)

print("🎵 Most Popular Playlist Tracks:")
df_popular_tracks

---

## 📝 Summary: SQL Cheat Sheet

### Basic Queries
```sql
-- Select all columns
SELECT * FROM table_name;

-- Select specific columns
SELECT col1, col2 FROM table_name;

-- Filter rows
SELECT * FROM table_name WHERE condition;

-- Sort results
SELECT * FROM table_name ORDER BY col1 DESC;

-- Limit results
SELECT * FROM table_name LIMIT 10;
```

### JOINs
```sql
-- Inner join (matching rows only)
SELECT * FROM table1
INNER JOIN table2 ON table1.id = table2.foreign_id;

-- Left join (all from left + matches from right)
SELECT * FROM table1
LEFT JOIN table2 ON table1.id = table2.foreign_id;
```

### Aggregation
```sql
-- Group and aggregate
SELECT category, COUNT(*), SUM(amount), AVG(price)
FROM table_name
GROUP BY category
HAVING COUNT(*) > 5
ORDER BY SUM(amount) DESC;
```

### Python Integration
```python
# Safe connection with context manager
with sqlite3.connect('database.db') as conn:
    df = pd.read_sql("SELECT * FROM table", conn)
    
# Parameterized queries (ALWAYS use these!)
df = pd.read_sql("SELECT * FROM table WHERE col = ?", conn, params=(value,))
```

---

## 🎉 Great Work!

You've learned the fundamentals of SQL and how to integrate it with Python and Pandas!

### Next Steps

1. **Practice More:** Try writing your own queries against the test database
2. **Assignment 9:** Apply these skills to build a magazine subscription database
3. **Week 10:** Learn advanced SQL features (subqueries, window functions)

### 📚 Resources

- [SQLite Documentation](https://www.sqlite.org/docs.html)
- [Pandas read_sql() Documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html)
- [SQL Tutorial (W3Schools)](https://www.w3schools.com/sql/)
- [SQLBolt Interactive Tutorial](https://sqlbolt.com/)